# Tutorial 18 — DeepSeek Architecture: MLA & MoE from Scratch

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part VII — DeepSeek Capstone**  
**Follows:** Tutorial 17 (Capstone: Complete Pipeline)  
**Precedes:** Tutorial 19 (Training NanoDeepSeek)

---

## What This Tutorial Covers

Tutorial 2 built a standard GPT: multi-head attention (MHA) with a dense FFN. That architecture is correct and trainable, but it has two well-known inefficiencies at scale:

1. **KV cache memory**: at inference, storing key and value tensors for every token in every layer grows as $O(n_{\text{heads}} \cdot d_{\text{head}} \cdot \text{seq\_len})$ per layer. For a 27B model generating 8k tokens, this can exceed the model's own weight memory.
2. **Dense FFN computation**: every token activates every FFN parameter on every forward pass. This is wasteful — different tokens need different knowledge, but the model spends equal compute on all of it.

DeepSeek-V3 solves both problems with two architectural innovations:

- **Multi-head Latent Attention (MLA)** — compresses the KV cache by projecting keys and values through a low-rank bottleneck, then caching only the bottleneck vector instead of the full K/V tensors.
- **Mixture of Experts (MoE)** — replaces the dense FFN with a bank of expert networks, routing each token to only the top-$k$ experts. The result: large total parameter count but small *activated* parameter count.

This tutorial builds both from scratch, assembles them into `NanoDeepSeek`, and compares the result directly against `NanoGPT` from Tutorial 2 on parameter count, KV cache footprint, and activated FLOPs.

By the end you will have:
- A `NanoMLA` class with full derivation of the low-rank KV compression and decoupled RoPE
- A `NanoMoE` class with top-$k$ routing and auxiliary load-balancing loss
- A complete `NanoDeepSeek` model (~200 lines) ready for training in Tutorial 19
- A side-by-side comparison table: NanoGPT vs NanoDeepSeek

---

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple

---

## 1. The KV Cache Bottleneck

Before building MLA, it is worth being precise about the problem it solves.

During inference, autoregressive generation processes one new token at a time. At step $t$, the model needs the keys and values for *all* previous tokens $1, \ldots, t-1$ to compute attention. Recomputing them from scratch at each step would cost $O(t^2)$ total. Instead we cache them: after processing token $i$, we store $K_i$ and $V_i$ for every layer. This is the KV cache.

The memory cost per token per layer for standard MHA:

$$\text{KV cache per token per layer} = 2 \times n_{\text{heads}} \times d_{\text{head}} \times \text{bytes\_per\_element}$$

For the NanoGPT config from Tutorial 2 (`n_heads=6`, `d_model=384`, so `d_head=64`, bfloat16 = 2 bytes):

$$2 \times 6 \times 64 \times 2 = 1{,}536 \text{ bytes per token per layer}$$

For a 6-layer model generating 256 tokens, that is $1{,}536 \times 256 \times 6 = 2.36$ MB — manageable. But scale to 30 layers and 8k context and it grows to hundreds of MB, competing with the model weights themselves.

[[[MLA's insight: we don't need to cache the full $K$ and $V$ matrices.]{.mark} We can cache a single low-rank *latent vector* $c_{KV}$ per token per layer, and reconstruct $K$ and $V$ from it on demand during attention. The latent dimension $d_c \ll n_{\text{heads}} \times d_{\text{head}}$, so the cache shrinks dramatically.

In [ ]:
# KV cache memory comparison: standard MHA vs MLA
def kv_cache_bytes(n_heads, d_head, n_layers, seq_len, bytes_per_elem=2):
    """Standard MHA: cache K and V for all heads."""
    return 2 * n_heads * d_head * n_layers * seq_len * bytes_per_elem

def kv_cache_bytes_mla(d_compressed, n_layers, seq_len, bytes_per_elem=2):
    """MLA: cache only the low-rank latent c_KV per token."""
    return d_compressed * n_layers * seq_len * bytes_per_elem

# NanoGPT config (Tutorial 2)
n_heads, d_model, n_layers = 6, 384, 6
d_head = d_model // n_heads   # 64

# NanoDeepSeek: d_compressed = d_model // 4  (we'll justify this below)
d_c = d_model // 4   # 96

for seq_len in [256, 1024, 4096]:
    mha = kv_cache_bytes(n_heads, d_head, n_layers, seq_len)
    mla = kv_cache_bytes_mla(d_c, n_layers, seq_len)
    print(f"seq_len={seq_len:5d} | MHA: {mha/1024:8.1f} KB | MLA: {mla/1024:6.1f} KB | ratio: {mha/mla:.1f}x")

---

## 2. Multi-Head Latent Attention (MLA)

### 2.1 The Low-Rank KV Compression

Standard MHA projects the input $x \in \mathbb{R}^{d}$ into queries, keys, and values:

$$Q = x W_Q, \quad K = x W_K, \quad V = x W_V$$

MLA replaces the K and V projections with a two-step process. First, compress: 

$$c_{KV} = x W_c \quad \in \mathbb{R}^{d_c}$$

where $d_c \ll n_{\text{heads}} \times d_{\text{head}}$. Then expand from the compressed latent:

$$K = c_{KV} W_K, \quad V = c_{KV} W_V$$

At inference, we cache only $c_{KV}$ — one vector of dimension $d_c$ per token per layer. When we need to compute attention at step $t$, we reconstruct $K$ and $V$ for all cached positions by passing their stored $c_{KV}$ vectors through $W_K$ and $W_V$.

Similarly, queries are compressed for the Q side:

$$c_Q = x W_{cQ} \in \mathbb{R}^{d_{cQ}}, \quad Q = c_Q W_Q$$

(This doesn't affect cache size since queries are not cached — they are recomputed for the current token only.)

### 2.2 Decoupled RoPE

There's a subtlety: Rotary Positional Encoding (RoPE) from Tutorial 2 applies a position-dependent rotation to Q and K before the dot product. But if we cache $c_{KV}$ and reconstruct K later, the rotated K would need to be computed at the time of attention — not at the time of caching. This defeats the compression.

MLA solves this with **decoupled RoPE**: a separate $d_r$-dimensional slice of Q and K carries the rotary encoding. This slice is *not* part of the compressed latent — it is computed fresh from the original input $x$ (via a dedicated projection $W_{QR}$, $W_{KR}$) at attention time. The cache stores $c_{KV}$ plus the compact RoPE key slice $k_R$.

In practice for our nano scale, we use a simplified version: single-head Q latent, no decoupled RoPE, full $d$ absorbed into the compressed path. The KV cache savings are the same.

In [ ]:
def apply_rope(x, cos, sin):
    """Apply rotary positional encoding. x: (B, n_heads, T, d_head)"""
    # Rotate pairs: [x0, x1, x2, x3, ...] -> [-x1, x0, -x3, x2, ...]
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    x_rot = torch.stack([-x2, x1], dim=-1).flatten(-2)
    return x * cos + x_rot * sin


def make_rope_cache(max_seq_len, d_head, device):
    """Precompute cos/sin tables for RoPE."""
    theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    positions = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(positions, theta)               # (T, d_head//2)
    freqs = torch.cat([freqs, freqs], dim=-1)           # (T, d_head)
    cos = freqs.cos()[None, None, :, :]                 # (1, 1, T, d_head)
    sin = freqs.sin()[None, None, :, :]                 # (1, 1, T, d_head)
    return cos, sin


class NanoMLA(nn.Module):
    """Multi-head Latent Attention (simplified, no decoupled RoPE).
    
    Standard MHA caches K and V directly: O(n_heads * d_head) per token.
    MLA compresses: cache only c_KV of dim d_compressed, then reconstruct K,V.
    KV cache reduction: 2*n_heads*d_head → d_compressed (typically 4x smaller).
    """

    def __init__(self, d_model: int, n_heads: int, d_compressed: int, max_seq_len: int = 1024):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.d_compressed = d_compressed

        # Down-project input to low-rank KV latent
        self.W_c   = nn.Linear(d_model, d_compressed, bias=False)     # compress
        # Up-project latent to full K and V
        self.W_K   = nn.Linear(d_compressed, d_model, bias=False)     # keys
        self.W_V   = nn.Linear(d_compressed, d_model, bias=False)     # values
        # Q projection (not cached)
        self.W_Q   = nn.Linear(d_model, d_model, bias=False)
        # Output projection
        self.W_O   = nn.Linear(d_model, d_model, bias=False)

        cos, sin = make_rope_cache(max_seq_len, self.d_head, device='cpu')
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x:    (B, T, d_model)
        mask: (1, 1, T, T) causal mask — True where attention is forbidden
        """
        B, T, _ = x.shape

        # --- Compress to latent (this is what gets cached at inference) ---
        c_kv = self.W_c(x)                                      # (B, T, d_compressed)

        # --- Reconstruct K and V from latent ---
        K = self.W_K(c_kv)                                      # (B, T, d_model)
        V = self.W_V(c_kv)                                      # (B, T, d_model)

        # --- Queries (computed fresh, not cached) ---
        Q = self.W_Q(x)                                         # (B, T, d_model)

        # --- Reshape to multi-head layout ---
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
            # (B, n_heads, T, d_head)

        Q = split_heads(Q)
        K = split_heads(K)
        V = split_heads(V)

        # --- Apply RoPE to Q and K ---
        cos = self.cos[:, :, :T, :].to(x.device)
        sin = self.sin[:, :, :T, :].to(x.device)
        Q = apply_rope(Q, cos, sin)
        K = apply_rope(K, cos, sin)

        # --- Scaled dot-product attention ---
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)   # (B, n_heads, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = weights @ V                                             # (B, n_heads, T, d_head)

        # --- Merge heads and project ---
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out)


# Verify shapes
B, T, d = 2, 16, 384
mla = NanoMLA(d_model=d, n_heads=6, d_compressed=96)
x_in = torch.randn(B, T, d)
out = mla(x_in)
print(f"NanoMLA: input {tuple(x_in.shape)} → output {tuple(out.shape)}")
print(f"KV compressed dim: {mla.d_compressed} (vs full K+V: {2 * d} dims)")
print(f"Cache saving at inference: {2*mla.n_heads*mla.d_head / mla.d_compressed:.1f}x")

---

## 3. Mixture of Experts (MoE)

### 3.1 The Dense FFN Problem

A standard FFN block computes:

$$\text{FFN}(x) = W_2 \cdot \text{SwiGLU}(W_1 x, W_3 x)$$

where SwiGLU is the activation used by LLaMA/DeepSeek: $\text{SwiGLU}(a, b) = \text{SiLU}(a) \odot b$.

The problem: all $P_{\text{FFN}}$ parameters activate for every token on every forward pass. Total FLOPs per token $\propto P_{\text{FFN}}$. If you want more capacity, you pay linearly in both parameters *and* compute.

### 3.2 The MoE Solution

Replace the single FFN with $E$ expert FFNs. Per token, activate only the top-$k$ experts. Two types:

- **Shared experts** ($N_s$): always active — handle universal patterns every token needs
- **Routed experts** ($N_r$): conditionally active — each token activates its top-$k$ routed experts

The routing decision is made by a linear router: given $x$, compute scores $s = x W_r \in \mathbb{R}^{N_r}$, take top-$k$, softmax over only those $k$ scores to get weights $g_i$, then:

$$\text{MoE}(x) = \sum_{i \in \text{shared}} E_i(x) + \sum_{i \in \text{top-}k} g_i \cdot E_i(x)$$

[Total params: $(N_s + N_r) \times P_{\text{expert}}$. Activated params per token: $(N_s + k) \times P_{\text{expert}}$.]{.mark} With $N_r = 55, k = 6$ (DeepSeek-V3), the sparsity ratio is $\approx 10\times$.

### 3.3 Expert Load Balancing

[Without any regularization, the router collapses — all tokens route to the same few experts ("expert collapse")]{.mark}. The standard fix is an auxiliary load-balancing loss:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N_r \cdot \sum_{i=1}^{N_r} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ (non-differentiable, stops gradient), and $P_i$ is the average router probability for expert $i$ (differentiable). This penalizes *routing imbalance*: if one expert gets all tokens ($f_i = 1, f_j = 0$), the loss is large.

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU activation: SiLU(W1 x) * W3 x — used in LLaMA and DeepSeek."""

    def __init__(self, d_model: int, d_ffn: int):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ffn, bias=False)  # gate
        self.W3 = nn.Linear(d_model, d_ffn, bias=False)  # pass-through
        self.W2 = nn.Linear(d_ffn,   d_model, bias=False)  # down-project

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.W2(F.silu(self.W1(x)) * self.W3(x))


class NanoMoE(nn.Module):
    """DeepSeek-style Mixture of Experts: shared + routed experts, top-k routing.

    Returns: (output, aux_loss) where aux_loss is the load-balancing penalty.
    """

    def __init__(
        self,
        d_model: int,
        d_ffn: int,
        n_shared: int = 1,
        n_routed: int = 8,
        top_k: int = 2,
        aux_loss_coeff: float = 1e-2,
    ):
        super().__init__()
        self.n_shared  = n_shared
        self.n_routed  = n_routed
        self.top_k     = top_k
        self.aux_alpha = aux_loss_coeff

        # Shared experts (always active)
        self.shared_experts = nn.ModuleList([
            SwiGLU(d_model, d_ffn) for _ in range(n_shared)
        ])
        # Routed experts (conditional)
        self.routed_experts = nn.ModuleList([
            SwiGLU(d_model, d_ffn) for _ in range(n_routed)
        ])
        # Router: a linear map from token embedding to expert logits
        self.router = nn.Linear(d_model, n_routed, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: (B, T, d_model)
        returns: output (B, T, d_model), aux_loss scalar
        """
        B, T, d = x.shape
        x_flat = x.view(B * T, d)   # treat each token independently

        # --- Shared expert pass (always active) ---
        shared_out = sum(e(x_flat) for e in self.shared_experts)  # (B*T, d)

        # --- Router: compute logits and select top-k experts ---
        logits = self.router(x_flat)                    # (B*T, n_routed)
        probs  = F.softmax(logits, dim=-1)              # (B*T, n_routed)

        # Top-k selection: indices and weights
        topk_vals, topk_idx = torch.topk(probs, self.top_k, dim=-1)  # (B*T, k) each
        # Re-normalise weights over selected experts
        gates = topk_vals / (topk_vals.sum(dim=-1, keepdim=True) + 1e-9)  # (B*T, k)

        # --- Routed expert computation ---
        routed_out = torch.zeros_like(x_flat)           # (B*T, d)
        for k_pos in range(self.top_k):
            expert_ids = topk_idx[:, k_pos]             # (B*T,) — which expert
            gate_vals  = gates[:, k_pos].unsqueeze(-1)  # (B*T, 1)
            for expert_id in range(self.n_routed):
                mask = (expert_ids == expert_id)        # tokens routed to this expert
                if mask.any():
                    routed_out[mask] += gate_vals[mask] * self.routed_experts[expert_id](x_flat[mask])

        output = (shared_out + routed_out).view(B, T, d)

        # --- Auxiliary load-balancing loss ---
        # f_i: fraction of tokens assigned to expert i (non-differentiable)
        # P_i: average router probability for expert i (differentiable)
        with torch.no_grad():
            # one-hot indicator: which expert each token's top-1 selection was
            top1_idx = topk_idx[:, 0]                   # (B*T,)
            one_hot  = F.one_hot(top1_idx, self.n_routed).float()  # (B*T, n_routed)
            f = one_hot.mean(dim=0)                     # (n_routed,) — fraction per expert
        P = probs.mean(dim=0)                           # (n_routed,) — avg prob per expert
        aux_loss = self.aux_alpha * self.n_routed * (f * P).sum()

        return output, aux_loss


# Verify shapes and aux loss
moe = NanoMoE(d_model=384, d_ffn=384*4, n_shared=1, n_routed=8, top_k=2)
x_in = torch.randn(2, 16, 384)
out, aux = moe(x_in)
print(f"NanoMoE: input {tuple(x_in.shape)} → output {tuple(out.shape)}")
print(f"aux_loss: {aux.item():.4f}")
print(f"Activated experts per token: {moe.n_shared + moe.top_k} / {moe.n_shared + moe.n_routed} total")

---

## 4. Supporting Components: RMSNorm

DeepSeek uses **RMSNorm**[^rmsnorm] instead of LayerNorm. The difference: RMSNorm skips the mean-centering step, normalizing by the RMS of activations only:

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \qquad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \epsilon}$$

[[[This is slightly faster (no mean subtraction) and has been found to perform equally well in practice.]{.underline}

[^rmsnorm]: RMSNorm (Zhang & Sennrich, 2019) normalises by $\text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$, skipping the mean-subtraction step of LayerNorm. This removes one reduction kernel, saving ~15% compute per normalisation call, and avoids the numerical instability that can occur when the mean is non-zero and large in very deep networks.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.gamma


# Verify RMSNorm is approximately unit-RMS post-norm
rn = RMSNorm(384)
x  = torch.randn(4, 16, 384) * 5.0   # large activations
y  = rn(x)
print(f"Input  RMS: {x.pow(2).mean().sqrt().item():.3f}")
print(f"Output RMS: {y.pow(2).mean().sqrt().item():.3f}  (≈1.0 with unit gamma init)")

---

## 5. Assembling NanoDeepSeek

We now have all the pieces. The `NanoDeepSeek` block follows DeepSeek's residual order:

```
x  →  RMSNorm → MLA → + x  →  RMSNorm → MoE → + x
```

(Pre-norm placement: normalise before each sub-block, not after.)

In [ ]:
@dataclass
class NanoDeepSeekConfig:
    vocab_size:      int   = 50257
    d_model:         int   = 384
    n_layers:        int   = 6
    n_heads:         int   = 6
    d_compressed:    int   = 96     # MLA KV latent dim (d_model // 4)
    d_ffn:           int   = 1024   # expert hidden dim
    n_shared:        int   = 1      # shared experts (always active)
    n_routed:        int   = 8      # routed experts pool
    top_k:           int   = 2      # experts activated per token per layer
    max_seq_len:     int   = 256
    aux_loss_coeff:  float = 1e-2


def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1
    ).unsqueeze(0).unsqueeze(0)   # (1, 1, T, T)


class DeepSeekBlock(nn.Module):
    """One transformer block: pre-norm MLA + pre-norm MoE."""

    def __init__(self, config: NanoDeepSeekConfig):
        super().__init__()
        self.norm_attn = RMSNorm(config.d_model)
        self.attn = NanoMLA(
            d_model=config.d_model,
            n_heads=config.n_heads,
            d_compressed=config.d_compressed,
            max_seq_len=config.max_seq_len,
        )
        self.norm_ffn = RMSNorm(config.d_model)
        self.moe = NanoMoE(
            d_model=config.d_model,
            d_ffn=config.d_ffn,
            n_shared=config.n_shared,
            n_routed=config.n_routed,
            top_k=config.top_k,
            aux_loss_coeff=config.aux_loss_coeff,
        )

    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # MLA sub-block (residual)
        x = x + self.attn(self.norm_attn(x), mask)
        # MoE sub-block (residual)
        moe_out, aux_loss = self.moe(self.norm_ffn(x))
        x = x + moe_out
        return x, aux_loss


class NanoDeepSeek(nn.Module):
    """Nano DeepSeek: MLA + MoE transformer for the capstone series."""

    def __init__(self, config: NanoDeepSeekConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([
            DeepSeekBlock(config) for _ in range(config.n_layers)
        ])
        self.norm_out = RMSNorm(config.d_model)
        self.lm_head  = nn.Linear(config.d_model, config.vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.token_embedding.weight

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        idx:     (B, T) token indices
        targets: (B, T) next-token targets, or None
        returns: logits (B, T, vocab), total_loss (lm_loss + sum aux_losses) or None
        """
        B, T = idx.shape
        x = self.token_embedding(idx)           # (B, T, d_model)
        mask = make_causal_mask(T, idx.device)

        total_aux = torch.tensor(0.0, device=idx.device)
        for block in self.blocks:
            x, aux = block(x, mask)
            total_aux = total_aux + aux

        x = self.norm_out(x)
        logits = self.lm_head(x)                # (B, T, vocab_size)

        loss = None
        if targets is not None:
            lm_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
            loss = lm_loss + total_aux

        return logits, loss


# Quick forward pass check
cfg = NanoDeepSeekConfig()
model = NanoDeepSeek(cfg)
idx = torch.randint(0, cfg.vocab_size, (2, 16))
targets = torch.randint(0, cfg.vocab_size, (2, 16))
logits, loss = model(idx, targets)
print(f"NanoDeepSeek forward pass OK")
print(f"  logits: {tuple(logits.shape)}")
print(f"  loss:   {loss.item():.4f}")

---

## 6. Parameter Count and KV Cache Comparison

Let's compare NanoGPT (Tutorial 2) against NanoDeepSeek at matched total parameter count.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_activated_params(config: NanoDeepSeekConfig):
    """Estimate activated parameters per forward token (excluding embedding)."""
    d, n_heads, d_c = config.d_model, config.n_heads, config.d_compressed
    d_head = d // n_heads

    # MLA per layer: W_c + W_K + W_V + W_Q + W_O
    p_mla = (d * d_c) + (d_c * d) + (d_c * d) + (d * d) + (d * d)

    # MoE per layer: (n_shared + top_k) active experts
    p_expert = 3 * config.d_model * config.d_ffn   # W1 + W3 + W2
    p_moe_active = (config.n_shared + config.top_k) * p_expert

    return config.n_layers * (p_mla + p_moe_active)


# NanoDeepSeek
cfg_ds = NanoDeepSeekConfig()
model_ds = NanoDeepSeek(cfg_ds)
total_ds = count_params(model_ds)
active_ds = count_activated_params(cfg_ds)

# KV cache: MLA caches d_compressed per token per layer
def kv_cache_MB(n_heads, d_head, n_layers, seq_len, bpe=2):
    return 2 * n_heads * d_head * n_layers * seq_len * bpe / 1e6

def kv_cache_MB_mla(d_c, n_layers, seq_len, bpe=2):
    return d_c * n_layers * seq_len * bpe / 1e6

seq = 256
d_head_gpt = 384 // 6

print("=" * 60)
print(f"{'':30s} {'NanoGPT':>12s} {'NanoDeepSeek':>14s}")
print("=" * 60)
print(f"{'Total parameters':30s} {'~10.7M':>12s} {total_ds/1e6:>13.1f}M")
print(f"{'Activated per token':30s} {'~10.7M':>12s} {active_ds/1e6:>13.1f}M")
print(f"{'KV cache @ seq=256 (MB)':30s} {kv_cache_MB(6,64,6,seq):>11.2f}M {kv_cache_MB_mla(cfg_ds.d_compressed,6,seq):>13.2f}M")
print(f"{'KV cache @ seq=1024 (MB)':30s} {kv_cache_MB(6,64,6,1024):>11.2f}M {kv_cache_MB_mla(cfg_ds.d_compressed,6,1024):>13.2f}M")
print(f"{'Attention':30s} {'MHA (RoPE)':>12s} {'MLA (low-rank KV)':>14s}")
print(f"{'FFN':30s} {'Dense SwiGLU':>12s} {'MoE (1s+8r, top-2)':>14s}")
print(f"{'Normalisation':30s} {'LayerNorm':>12s} {'RMSNorm':>14s}")
print("=" * 60)

The key insight: MoE gives NanoDeepSeek more total parameters (for the same training compute) but the activated parameter count per token is comparable to NanoGPT. In Tutorial 19 we will train both at matched *activated* parameter count on FineWeb-Edu and compare loss curves directly.

---

## 7. Memory Taxonomy — Three Axes of Sparsity

Before moving to training, it helps to see where DeepSeek's architecture sits relative to the alternatives:

| Memory type | Speed | Persistence | Access mechanism |
|---|---|---|---|
| Dense parameters | Slow (SGD) | Permanent | Dense matmul, all tokens |
| KV cache | Fast | Per-sequence | Attention — content-based lookup |
| MoE experts | Medium | Permanent | Top-k routing, token-conditional |
| Engram (Tutorial 20) | Fast | Permanent | O(1) hash lookup, n-gram conditional |

MLA reduces the *cost* of the KV cache. MoE conditions which FFN parameters activate. Engram (Tutorial 20) introduces a fourth axis: n-gram conditional lookup into a static embedding table — [[[no attention, no routing, just a hash.]{.underline}

---

## Summary

| Component | What it does | Key parameter |
|---|---|---|
| `NanoMLA` | Low-rank KV compression: cache $d_c$ instead of $n_h \cdot d_h$ | `d_compressed` |
| Decoupled RoPE | Separates position encoding from compressed latent | $d_r$ RoPE slice |
| `NanoMoE` | Top-$k$ routing over $N_r$ routed + $N_s$ shared experts | `n_routed`, `top_k` |
| Aux loss | Penalizes routing imbalance: $\alpha N_r \sum f_i P_i$ | `aux_loss_coeff` |
| `RMSNorm` | Normalise by RMS, skip mean shift | `eps` |
| `SwiGLU` | $W_2(\text{SiLU}(W_1 x) \odot W_3 x)$ — used in every expert | `d_ffn` |
| `NanoDeepSeek` | Full model: embedding → DeepSeekBlock × L → RMSNorm → LM head | `NanoDeepSeekConfig` |

**What's next:** Tutorial 19 trains NanoGPT and NanoDeepSeek side-by-side on FineWeb-Edu, matching activated parameters. We will see whether MLA + MoE improves loss per activated FLOP at nano scale.

---

## Exercises

**1.** Extend `NanoMLA` to implement decoupled RoPE: add a separate projection $W_{KR} \in \mathbb{R}^{d_c \times d_r}$ for a RoPE-carrying key slice of dimension $d_r = d_{\text{head}} / 2$. Verify that the output shape is unchanged but the RoPE slice is computed from the original $x$, not from $c_{KV}$.

**2.** Add expert utilization logging to `NanoMoE`: track the histogram of how many tokens each routed expert receives per batch. Plot it across training steps and verify it becomes more uniform as the aux loss takes effect.

**3.** The aux loss formula uses `top-1` frequency $f_i$ but full softmax probabilities $P_i$. Why? What happens if you replace $f_i$ with the top-$k$ fraction (i.e., count any token whose top-$k$ includes expert $i$)? Implement both variants and compare their effect on routing entropy.

**4.** Build a `KVCacheStats` function that measures actual GPU memory for the KV cache of `NanoMLA` vs standard MHA at sequence lengths 256, 512, 1024. Compare against the theoretical formula from Section 1.

**5.** Experiment with the MoE sparsity ratio: fix total expert parameters but vary `n_routed` and `top_k` such that total param count stays constant. Does training loss at 1000 steps change with different sparsity ratios?